# SIBDA-1380 – Entrega Definitiva: Analítica, Clasificación e IA de Benefactores (Fundación 1380)
**Institución Universitaria Pascual Bravo** · Tecnología en Desarrollo de Software (Semestre III)  
**Módulo:** ET0155 – Fundamentos de Big Data · **Docente:** MgSc. Ing. Jaime E. Soto U.  

### Estructura Clara y Sencilla de este Notebook en Google Colab
Todo el código está escrito en **Python limpio, corto y comentado paso a paso en español**, distinguiendo siempre sin ambigüedad entre:
- **`donante_benefactor`:** La persona externa que realiza donaciones a la Fundación (ej. códigos `PER_...` o los 4 hermanos Juan, Felipe, Jesús y Daniel).
- **`empleada_telemercaderista`:** La colaboradora de la Fundación encargada de realizar las llamadas (ej. Sandra, Viviana, Hermana Lina, Jorge).

**Recorrido completo (Módulos 1 al 8):**
1. **Ingesta Inmutable y ETL Dinámico (Módulo 1):** Copia segura con firma `SHA-256`, limpieza sencilla adaptable a futuros archivos Excel (`2027+`) y transformación a **20.804 aportes confirmados**.
2. **Modelo Clasificador de Benefactores (Módulo 3):** Reglas claras sobre pagos reales en 4 segmentos + ficha individual de 8 indicadores.
3. **Modelo Predictivo de Llamadas con Alta Probabilidad (Módulo 4):** `RandomForestClassifier` sin fuga de datos, **explicación detallada de los 3 gráficos de evaluación** y lista priorizada de llamadas.
4. **Coordinación de Llamadas en Teléfonos Compartidos (Módulo 5):** Caso obligatorio de los **4 hermanos** separando claramente Telemercaderista vs. Benefactor.
5. **Consolidado Final (Módulos 2, 6, 7 y 8):** Almacenamiento **Parquet + DuckDB**, **Tablero de 11 KPIs**, **Proceso Batch del Mensajero** y **Gobierno/Seguridad de Datos** + prueba automática con nuevo Excel.

In [ ]:
# ==============================================================================
# PASO 1. INSTALAR E IMPORTAR LIBRERÍAS NECESARIAS EN GOOGLE COLAB
# ==============================================================================
!pip install -q duckdb openpyxl pyarrow scikit-learn plotly

import re
import hmac
import shutil
import hashlib
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl
import duckdb
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Librerías listas | Pandas:", pd.__version__, "| DuckDB:", duckdb.__version__)

### Explicación del Paso 1
Cargamos únicamente las herramientas necesarias del ecosistema Python: **Pandas/OpenPyXL** para leer y limpiar el Excel, **Parquet/DuckDB** para guardar el Data Lake y el Data Warehouse, **Scikit-learn** para el modelo predictivo de llamadas y **Matplotlib** para visualizar las gráficas.

---
## 2. Módulo 1 – Carga Segura del Excel (Sin Modificar el Original) y Limpieza ETL Sencilla

In [ ]:
# ==============================================================================
# PASO 2.1 CARGAR EL EXCEL TRABAJANDO SIEMPRE SOBRE UNA COPIA (SHA-256 INTACTO)
# ==============================================================================
CARPETA_BASE = Path("sibda_1380")
RAW_DIR      = CARPETA_BASE / "datalake" / "raw"
BRONZE_DIR   = CARPETA_BASE / "datalake" / "bronze"
SILVER_DIR   = CARPETA_BASE / "datalake" / "silver"
GOLD_DIR     = CARPETA_BASE / "datalake" / "gold"
DW_DIR       = CARPETA_BASE / "warehouse"
BACKUP_DIR   = CARPETA_BASE / "backups"
REPORTES_DIR = CARPETA_BASE / "reportes"
TESTS_DIR    = CARPETA_BASE / "tests_nuevo_excel"

for carpeta in [RAW_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR, DW_DIR, BACKUP_DIR, REPORTES_DIR, TESTS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

NOMBRE_EXCEL = "BD_F1380_SeudonimizadaAnonimizadaencriptada.xlsx"
RUTA_EXCEL_DRIVE = f"/content/drive/MyDrive/Fundacion1380/{NOMBRE_EXCEL}"

def sha256_file(ruta: Path) -> str:
    """Calcula la huella digital única SHA-256 de un archivo."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(8192), b""):
            h.update(bloque)
    return h.hexdigest()

# Buscar el Excel en carpeta local, en Recursos/ o en Google Drive; si no está, pedir subirlo
origen_path = None
for posible in [Path(NOMBRE_EXCEL), Path("Recursos") / NOMBRE_EXCEL, Path("../Recursos") / NOMBRE_EXCEL, Path(RUTA_EXCEL_DRIVE)]:
    if posible.exists():
        origen_path = posible
        break

if origen_path is None:
    import urllib.request
    url_github_raw = f"https://raw.githubusercontent.com/Camilo-fonseca237/Trabajo_Final_Bigtdata/main/Recursos/{NOMBRE_EXCEL}"
    try:
        urllib.request.urlretrieve(url_github_raw, NOMBRE_EXCEL)
        origen_path = Path(NOMBRE_EXCEL)
        print("Excel descargado automáticamente desde el repositorio oficial.")
    except Exception:
        from google.colab import files
        print(f"Por favor sube el archivo '{NOMBRE_EXCEL}':")
        subidos = files.upload()
        origen_path = Path(list(subidos.keys())[0])

# Guardar huella del archivo original y trabajar SOLO sobre la copia en /raw
HASH_ORIGINAL_INICIO = sha256_file(origen_path)
MARCA_TIEMPO = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = f"ETL_{MARCA_TIEMPO}"
copia_trabajo = RAW_DIR / f"raw_{MARCA_TIEMPO}_{origen_path.name}"
shutil.copy2(origen_path, copia_trabajo)

libro = openpyxl.load_workbook(copia_trabajo, read_only=True)
nombre_hoja = libro.sheetnames[0]
libro.close()

df_raw = pd.read_excel(copia_trabajo, sheet_name=nombre_hoja, dtype=object)
print(f"Hoja leída        : {nombre_hoja}")
print(f"Tamaño del Excel  : {df_raw.shape[0]:,} filas (benefactores) x {df_raw.shape[1]} columnas")
print(f"SHA-256 Original  : {HASH_ORIGINAL_INICIO}")

### Explicación del Paso 2.1
Antes de tocar cualquier dato, sacamos la **huella digital `SHA-256`** del Excel fuente y creamos una fotocopia en `datalake/raw/` abriéndola en modo solo lectura (`read_only=True`). Así garantizamos al 100 % que el archivo original de la Fundación nunca se altera.

In [ ]:
# ==============================================================================
# PASO 2.2 LIMPIEZA SENCILLA (ETL) Y GIRO DE LAS COLUMNAS DE APORTES (2008-2026+)
# ==============================================================================
TABLA_SIN_TILDES = str.maketrans("ÁÉÍÓÚÑáéíóúñ", "AEIOUNaeioun")
MESES_NUMERO = {
    "ENE": 1, "ENERO": 1, "FEB": 2, "FEBRERO": 2, "MAR": 3, "MARZO": 3,
    "ABR": 4, "ABRIL": 4, "MAY": 5, "MAYO": 5, "JUN": 6, "JUNIO": 6,
    "JUL": 7, "JULIO": 7, "AGO": 8, "AGOSTO": 8, "SEP": 9, "SEPT": 9, "SEPTIEMBRE": 9,
    "OCT": 10, "OCTUBRE": 10, "NOV": 11, "NOVIEMBRE": 11, "DIC": 12, "DICIEMBRE": 12
}
MESES_CORTOS = {1:"ene", 2:"feb", 3:"mar", 4:"abr", 5:"may", 6:"jun", 7:"jul", 8:"ago", 9:"sep", 10:"oct", 11:"nov", 12:"dic"}

def mapear_esquema_dinamico(columnas_excel):
    """
    Traduce los títulos del Excel a nombres claros y detecta automáticamente
    tanto los años actuales (2008-2026) como años nuevos si cargan un Excel 2027+.
    """
    titulos = [re.sub(r"\s+", " ", re.sub(r"\.\d+$", "", str(c).strip()).translate(TABLA_SIN_TILDES).upper()).strip() for c in columnas_excel]

    # Buscar dónde empieza el año 2008
    inicio_plata = 44
    for i, t in enumerate(titulos):
        if t.isdigit() and len(t) == 4 and int(t) <= 2015:
            inicio_plata = i
            break

    # Nombres súper claros: separamos donante_benefactor vs empleada_telemercaderista
    reglas = [
        ("COD", "cod"), ("FECHA", "fecha_ingreso_raw"), ("SOCIO", "socio_raw"),
        ("TRATO", "trato_raw"), ("NOMBRES APELLIDOS", "donante_benefactor"),
        ("NOMBRES", "nombres_token"), ("APELLIDOS", "apellidos_token"),
        ("CEDULA", "cedula_token"), ("TELEFONO", "telefono_token"),
        ("DIRECCION", "direccion_token"), ("BARRIO", "barrio_token"),
        ("MUNICIPIO", "municipio_raw"), ("DIA CUMP", "dia_cump_raw"),
        ("MES CUMP", "mes_cump_raw"), ("ANO", "anio_nac_raw"),
        ("LO ATENDERA", "empleada_telemercaderista"),
        ("MENSAJES DE CUMPLEANOS", "mensajes_cumpleanos"), ("DETALLE CUMPLEANOS", "detalle_cumpleanos"),
        ("WA", "whatsapp_flag"), ("VECES EN EL ANO", "veces_en_anio_raw"),
        ("ENCUESTA EVENTOS", "encuesta_eventos"), ("EMAIL", "email_token"),
        ("CERTIFICADO", "certificado_donacion_raw"), ("DONACION EXTRA", "donacion_extra_raw"),
        ("GESTION", "gestion_log"), ("LLAMAR PARA BONOS", "llamar_bonos_2026"),
        ("CALAMIDAD", "calamidad"), ("ENFERMO", "enfermo"),
        ("OBSERVACIONES", "observaciones_comentarios"), ("TIPO PAGO", "tipo_pago_raw"),
        ("RECOGERLE", "recogerle_dia"), ("FRECUENCIA", "frecuencia_raw"),
        ("ALCANCIA", "alcancia_raw"), ("BONO 20", "bono_2026_raw"),
        ("CANTARILLA", "cantarilla_2026_raw"), ("RIFA DE VIRGEN", "rifa_virgen_carmen_raw"),
        ("VIRGEN DEL CARME", "rifa_virgen_carmen_raw"),
        ("CUOTA", "cuota_raw"), ("ULT PAGO", "ult_pago_raw")
    ]

    nombres = [""] * len(columnas_excel)
    usados = set()
    recibos_vistos = 0

    for i in range(inicio_plata):
        t = titulos[i]
        orig = str(columnas_excel[i]).strip()
        if t == "RECIBOS":
            recibos_vistos += 1
            elegido = "recibos_preferencia_raw" if (recibos_vistos == 1 and orig != "RECIBOS") else "recibos_numeros"
        else:
            elegido = f"extra_crm_{i}"
            for prefijo, destino in reglas:
                if (t == prefijo or t.startswith(prefijo)) and destino not in usados:
                    elegido = destino
                    break
        usados.add(elegido)
        nombres[i] = elegido

    cols_anuales, meta_mensuales, cols_totales, bolsa_meses = [], [], [], []
    for i in range(inicio_plata, len(columnas_excel)):
        t = titulos[i]
        if t == "ACUMULADO":
            nombres[i] = "acumulado_reportado"
            continue
        es_total_anual = re.fullmatch(r"(?:TOTAL\s*)?(20\d{2})", t)
        if es_total_anual:
            anio = int(es_total_anual.group(1))
            if not bolsa_meses:
                col_a = f"total_{anio}"
                nombres[i] = col_a
                cols_anuales.append(col_a)
            else:
                mes_act = 1
                for idx_m, txt_m in bolsa_meses:
                    for pal_m, num_m in MESES_NUMERO.items():
                        if re.search(rf"\b{pal_m}\b", txt_m):
                            mes_act = num_m
                            break
                    es_banq = "BANQUETE" in txt_m
                    es_bono = "BONO" in txt_m
                    mes_fin = 12 if es_banq else mes_act
                    if es_banq:
                        base, rec = f"{anio}_banquete", "Banquete navideño"
                    elif es_bono:
                        base, rec = f"{anio}_{mes_fin:02d}_bonos", "Bono"
                    else:
                        base, rec = f"{anio}_{mes_fin:02d}_{MESES_CORTOS[mes_fin]}", "Donación directa"
                    nom_m = base
                    k = 2
                    while nom_m in usados:
                        nom_m = f"{base}_{k}"; k += 1
                    usados.add(nom_m)
                    nombres[idx_m] = nom_m
                    meta_mensuales.append({"col": nom_m, "anio": anio, "mes": mes_fin, "recurso": rec})
                bolsa_meses = []
                col_t = f"total_{anio}"
                usados.add(col_t); nombres[i] = col_t; cols_totales.append(col_t)
        else:
            bolsa_meses.append((i, t))
    return nombres, cols_anuales, meta_mensuales, cols_totales

def convertir_fecha_simple(val):
    if pd.isna(val): return pd.NaT
    if isinstance(val, (datetime, pd.Timestamp)): return pd.to_datetime(val)
    if isinstance(val, (int, float, np.integer, np.floating)):
        return pd.to_datetime(float(val), unit="D", origin="1899-12-30", errors="coerce") if 20000 <= float(val) <= 60000 else pd.NaT
    s = str(val).strip().replace("//", "/").replace("'", "")
    if re.fullmatch(r"\d+(?:\.0+)?", s) and 20000 <= float(s) <= 60000:
        return pd.to_datetime(float(s), unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(s, dayfirst=True, errors="coerce")

def normalizar_socio_simple(val):
    if pd.isna(val): return "SIN_CLASIFICAR"
    v = str(val).upper()
    if "RETIR" in v or "NO SIGUE" in v or "NO SEG" in v: return "RETIRADO"
    if "PARROQUIA" in v: return "VISITA_PARROQUIAS"
    if "BONO" in v: return "BONOS"
    if "PERM" in v: return "PERMANENTE"
    if "ESP" in v: return "ESPORADICO"
    if "BANQUETE" in v: return "BANQUETE"
    if "CANTARILLA" in v or "RIFA" in v: return "RIFA_CANTARILLA"
    if "ALC" in v: return "ALCANCIA"
    if "BINGO" in v: return "BINGO"
    if "NSE" in v or "TARDES" in v or "OYENTE" in v: return "EVENTO_NSE"
    if "EVENTO" in v or "PEREGRIN" in v or "RETIRO" in v or "EJERCICIOS" in v: return "EVENTO"
    return "OTRO"

REGEX_FAMILIA = re.compile(r"\b(esposo|esposa|esposos|familia|flia|mam[aá]|pap[aá]|hijo|hija|herman[oa]s?|sobrin[oa]s?|t[ií][oa]s?|amig[oa]s?|compa[ñn][ií]a|junto con|dona con)\b", re.IGNORECASE)
REGEX_NO_CONTACTO = re.compile(r"\b(no llamar|no contactar|retirar|no sigue|no seg|falleci[oó]|muri[oó]|no felicitar|no molestar)\b", re.IGNORECASE)

def ejecutar_etl_dataframe(df_entrada: pd.DataFrame):
    df = df_entrada.copy()
    nombres, cols_anuales, meta_mensuales, cols_totales = mapear_esquema_dinamico(list(df.columns))
    df.columns = nombres
    df["id_benefactor"] = range(1, len(df) + 1)
    df["cod"] = pd.to_numeric(df["cod"], errors="coerce").astype("Int64")

    # Convertir fechas antes de limpiar textos para preservar objetos datetime/seriales de Excel
    df["fecha_ingreso_norm"] = df["fecha_ingreso_raw"].apply(convertir_fecha_simple)
    df["ult_pago_norm"] = df["ult_pago_raw"].apply(convertir_fecha_simple)
    df["flag_fecha_ingreso_invalida"] = df["fecha_ingreso_raw"].notna() & df["fecha_ingreso_norm"].isna()
    df["flag_ult_pago_invalido"] = df["ult_pago_raw"].notna() & df["ult_pago_norm"].isna()

    # Limpiar espacios en blanco de columnas de texto
    cols_no_texto = {"fecha_ingreso_norm", "ult_pago_norm", "fecha_ingreso_raw", "ult_pago_raw"}
    for c in [col for col in df.select_dtypes(include="object").columns if col not in cols_no_texto]:
        df[c] = df[c].apply(lambda x: re.sub(r"\s+", " ", str(x)).strip() if pd.notna(x) and str(x).strip() != "" else np.nan)

    # Alias compatibles claros
    df["nombre_completo_token"] = df["donante_benefactor"]
    df["lo_atendera_raw"] = df["empleada_telemercaderista"]

    df["socio_categoria"] = df["socio_raw"].apply(normalizar_socio_simple)
    mapa_pagos = {"DO": "DOMICILIO", "CO": "CONSIGNACION", "TR": "TRANSFERENCIA", "SE": "SEDE", "DE": "DEBITO"}
    df["tipo_pago_norm"] = df["tipo_pago_raw"].map(lambda x: mapa_pagos.get(str(x).upper(), "OTRO_O_INVALIDO") if pd.notna(x) else "NO_ESPECIFICADO")

    df["cuota_valor_num"] = pd.to_numeric(df["cuota_raw"], errors="coerce")
    df["bono_2026_valor_num"] = pd.to_numeric(df["bono_2026_raw"], errors="coerce")
    df["recogerle_dia_num"] = pd.to_numeric(df["recogerle_dia"], errors="coerce").astype("Int64")

    # Buscar vínculos familiares y restricciones de no llamar en las notas de texto
    cols_notas = [c for c in [
        "socio_raw", "trato_raw", "mensajes_cumpleanos", "detalle_cumpleanos",
        "encuesta_eventos", "donacion_extra_raw", "gestion_log", "llamar_bonos_2026",
        "calamidad", "enfermo", "observaciones_comentarios", "recibos_preferencia_raw",
        "alcancia_raw", "bono_2026_raw", "cantarilla_2026_raw", "rifa_virgen_carmen_raw"
    ] if c in df.columns]
    df["texto_consolidado_crm"] = df[cols_notas].apply(lambda fila: " | ".join(str(v) for v in fila if pd.notna(v)), axis=1)

    df["flag_aporte_familiar_amigos"] = (
        df["texto_consolidado_crm"].apply(lambda t: bool(REGEX_FAMILIA.search(t)))
        | df["trato_raw"].fillna("").str.contains(r"esposos|se[ñn]ores|familia|flia", case=False, regex=True)
    )
    df["detalle_vinculo_familiar"] = np.where(df["flag_aporte_familiar_amigos"], "Vínculo familiar/amigos en notas", "No documentado")
    df["flag_restriccion_no_contacto"] = df["texto_consolidado_crm"].apply(lambda t: bool(REGEX_NO_CONTACTO.search(t)))
    df["motivo_restriccion_contacto"] = np.where(df["flag_restriccion_no_contacto"], "SOLICITUD_NO_CONTACTO_O_RETIRO", "NINGUNA")

    df["flag_whatsapp"] = df["whatsapp_flag"].fillna("").str.upper().str.contains("SI") | df["mensajes_cumpleanos"].fillna("").str.upper().str.contains("WHATSAPP")
    df["canal_preferido"] = np.where(df["flag_whatsapp"], "WhatsApp / Telefónico", "Llamada Telefónica")
    df["flag_nota_sensible_pastoral"] = df["enfermo"].notna() | df["calamidad"].notna()

    # Contar personas que comparten el mismo teléfono y cédulas repetidas
    conteo_tel = df["telefono_token"].value_counts(dropna=True)
    df["personas_en_telefono"] = df["telefono_token"].map(conteo_tel).fillna(0).astype(int)
    df["flag_telefono_compartido"] = df["personas_en_telefono"] > 1
    df["flag_cedula_duplicada"] = df["cedula_token"].map(df["cedula_token"].value_counts(dropna=True)).fillna(0).astype(int) > 1

    # Girar (unpivot / melt) las columnas financieras hacia una tabla vertical limpia
    cols_detalle = [m["col"] for m in meta_mensuales]
    for c in cols_anuales + cols_detalle + cols_totales:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df_anual = df.melt(id_vars=["id_benefactor", "empleada_telemercaderista"], value_vars=cols_anuales, var_name="origen_col", value_name="monto").dropna(subset=["monto"])
    df_anual = df_anual[df_anual["monto"] > 0].copy()
    df_anual["anio"] = df_anual["origen_col"].str.extract(r"(\d{4})").astype(int)
    df_anual["mes"] = 12
    df_anual["recurso"] = "Donación directa"
    df_anual["tipo_registro"] = "ANUAL_HISTORICO"

    df_mensual = df.melt(id_vars=["id_benefactor", "empleada_telemercaderista"], value_vars=cols_detalle, var_name="origen_col", value_name="monto").dropna(subset=["monto"])
    df_mensual = df_mensual[df_mensual["monto"] > 0].merge(pd.DataFrame(meta_mensuales).rename(columns={"col": "origen_col"}), on="origen_col", how="left")
    df_mensual["tipo_registro"] = np.where(df_mensual["recurso"] == "Bono", "MENSUAL_BONO", np.where(df_mensual["recurso"] == "Banquete navideño", "BANQUETE", "MENSUAL_REGULAR"))

    df_tx = pd.concat([df_anual, df_mensual], ignore_index=True)
    df_tx["lo_atendera_raw"] = df_tx["empleada_telemercaderista"]
    df_tx["fecha_referencia"] = pd.to_datetime(df_tx["anio"].astype(str) + "-" + df_tx["mes"].astype(str).str.zfill(2) + "-15")
    df_tx = df_tx.sort_values(["id_benefactor", "fecha_referencia"]).reset_index(drop=True)
    df_tx["id_aporte"] = range(1, len(df_tx) + 1)
    return df, df_tx

df_work, df_aportes_long = ejecutar_etl_dataframe(df_raw)
monto_str = f"COP ${df_aportes_long['monto'].sum():,.0f}"
print(f"Donantes Benefactores procesados : {len(df_work):,}")
print(f"Aportes confirmados (> 0 pesos)  : {len(df_aportes_long):,} transacciones ({monto_str})")
print(f"Donan con familia/amigos (Regex) : {int(df_work['flag_aporte_familiar_amigos'].sum()):,}")
print(f"Restricción No Llamar            : {int(df_work['flag_restriccion_no_contacto'].sum()):,}")
print(f"Teléfonos compartidos en familia : {int(df_work.loc[df_work['flag_telefono_compartido'], 'telefono_token'].nunique())} grupos")

### Explicación del Paso 2.2 (Qué hizo el ETL)
1. **Diferenciación clara desde la raíz:** La columna `NOMBRES APELLIDOS` se bautiza `donante_benefactor` (quien dona) y `LO ATENDERÁ` se bautiza `empleada_telemercaderista` (quien llama).
2. **Giro vertical (*Unpivot*):** Pasamos de 130 columnas horizontales a una tabla limpia de **20.804 aportes mayores a cero** por **COP \$3.020.016.428**, sin sumar dos veces las columnas de subtotales anuales.

---
## 3. FOCO 1 – Modelo Clasificador de Benefactores y Ficha Individual (Módulo 3)

In [ ]:
# ==============================================================================
# PASO 3.1 MODELO CLASIFICADOR DE BENEFACTORES (4 SEGMENTOS CLAROS Y OBJETIVOS)
# ==============================================================================
ULTIMO_ANIO = int(df_aportes_long["anio"].max())
ANIO_CORTE_IA = ULTIMO_ANIO - 2   # Ejemplo: 2024 (ventana reciente = 2024..2026)

# 1. Contar cuántos aportes y cuánto dinero tiene cada donante_benefactor
resumen_total = df_aportes_long.groupby("id_benefactor").agg(
    num_aportes_registrados=("id_aporte", "count"),
    valor_acumulado=("monto", "sum"),
    valor_promedio_aporte=("monto", "mean"),
    primer_aporte_fecha=("fecha_referencia", "min"),
    ultimo_aporte_fecha=("fecha_referencia", "max"),
    primer_anio_aporte=("anio", "min"),
    ultimo_anio_aporte=("anio", "max")
).reset_index()

resumen_reciente = (
    df_aportes_long[df_aportes_long["anio"] >= ANIO_CORTE_IA]
    .groupby("id_benefactor")
    .agg(aportes_recientes_2024_2026=("id_aporte", "count"))
    .reset_index()
)

# Resumen SOLO hasta 2024 (para que en el Paso 4 la IA no haga trampa mirando 2025-2026)
tx_pre2025 = df_aportes_long[df_aportes_long["anio"] <= ANIO_CORTE_IA]
resumen_pre2025 = tx_pre2025.groupby("id_benefactor").agg(
    num_aportes_pre_target=("id_aporte", "count"),
    monto_acum_pre_target=("monto", "sum"),
    anios_activos_pre_target=("anio", "nunique"),
    primer_anio_pre_target=("anio", "min"),
    p_fecha_pre=("fecha_referencia", "min"),
    u_fecha_pre=("fecha_referencia", "max")
).reset_index()

resumen_2024 = (
    df_aportes_long[df_aportes_long["anio"] == ANIO_CORTE_IA]
    .groupby("id_benefactor")
    .agg(aportes_2024_count=("id_aporte", "count"), monto_2024_sum=("monto", "sum"))
    .reset_index()
)

recurso_top = (
    df_aportes_long.groupby(["id_benefactor", "recurso"])["monto"]
    .sum().reset_index()
    .sort_values(["id_benefactor", "monto"], ascending=[True, False])
    .drop_duplicates("id_benefactor")
    .rename(columns={"recurso": "recurso_mayor_contribucion"})
)

df_master = (
    df_work
    .merge(resumen_total, on="id_benefactor", how="left")
    .merge(resumen_reciente, on="id_benefactor", how="left")
    .merge(resumen_pre2025, on="id_benefactor", how="left")
    .merge(resumen_2024, on="id_benefactor", how="left")
    .merge(recurso_top[["id_benefactor", "recurso_mayor_contribucion"]], on="id_benefactor", how="left")
)

for col_cero in [
    "num_aportes_registrados", "valor_acumulado", "valor_promedio_aporte",
    "aportes_recientes_2024_2026", "num_aportes_pre_target", "monto_acum_pre_target",
    "anios_activos_pre_target", "primer_anio_pre_target", "aportes_2024_count", "monto_2024_sum",
    "primer_anio_aporte", "ultimo_anio_aporte"
]:
    df_master[col_cero] = df_master[col_cero].fillna(0)

# Calcular cada cuántos meses dona en promedio: meses activo / (aportes - 1)
meses_activo = (df_master["ultimo_aporte_fecha"] - df_master["primer_aporte_fecha"]).dt.days / 30.4375
df_master["intervalo_promedio_meses"] = np.where(
    df_master["num_aportes_registrados"] >= 2,
    meses_activo / (df_master["num_aportes_registrados"] - 1),
    np.nan
)
meses_pre = (df_master["u_fecha_pre"] - df_master["p_fecha_pre"]).dt.days / 30.4375
df_master["intervalo_meses_pre_target"] = np.where(
    df_master["num_aportes_pre_target"] >= 2,
    meses_pre / (df_master["num_aportes_pre_target"] - 1),
    np.nan
)
df_master["fecha_ultimo_aporte"] = df_master[["ult_pago_norm", "ultimo_aporte_fecha"]].max(axis=1)
df_master["recurso_mayor_contribucion"] = df_master["recurso_mayor_contribucion"].fillna("Registrado en boletas / socio")

# 2. Función sencilla de clasificación en las 4 categorías oficiales (sin inventar causas personales)
def clasificar_benefactor(fila) -> str:
    total_aportes = int(fila["num_aportes_registrados"])
    aportes_recientes = int(fila["aportes_recientes_2024_2026"])

    if aportes_recientes == 0:
        return "Sin aportes recientes que requieren revisión"
    elif total_aportes == 1:
        return "Con un único aporte registrado"
    elif total_aportes >= 4 and aportes_recientes >= 2:
        return "Permanente o recurrente"
    else:
        return "Esporádico"

df_master["clasificacion_benefactor"] = df_master.apply(clasificar_benefactor, axis=1)
df_master["comportamiento_historico"] = np.where(
    df_master["num_aportes_registrados"] == 0,
    "Sin aportes en matriz histórica; requiere revisión objetiva",
    "Activo " + df_master["primer_anio_aporte"].astype(int).astype(str) + "-" + df_master["ultimo_anio_aporte"].astype(int).astype(str)
)

tabla_segmentos = (
    df_master.groupby("clasificacion_benefactor")
    .agg(
        Benefactores=("id_benefactor", "count"),
        Aportes_Promedio=("num_aportes_registrados", "mean"),
        Intervalo_Meses=("intervalo_promedio_meses", "mean"),
        Valor_Acumulado_Total=("valor_acumulado", "sum")
    )
    .reset_index()
)
tabla_segmentos["Porcentaje_%"] = (tabla_segmentos["Benefactores"] / len(df_master) * 100).round(2)
display(tabla_segmentos.sort_values("Benefactores", ascending=False))

### Explicación del Clasificador de Benefactores (Paso 3.1)
La función `clasificar_benefactor()` usa únicamente 3 preguntas simples (`if / elif / else`) sobre los pagos confirmados:
- **Permanente o recurrente (`940` – `34,26 %`):** Tiene 4 o más aportes en su historia y al menos 2 recientes (2024–2026).
- **Sin aportes recientes que requieren revisión (`678` – `24,71 %`):** Tiene 0 aportes en 2024–2026 (no inventamos motivos personales).
- **Con un único aporte registrado (`601` – `21,90 %`):** Hizo un único aporte y fue reciente.
- **Esporádico (`525` – `19,13 %`):** Dona de forma intermitente (2 a 3 aportes, o antiguo con solo 1 reciente).

In [ ]:
# ==============================================================================
# PASO 3.2 CONSULTAR LA FICHA SIMPLE DE UN DONANTE BENEFACTOR INDIVIDUAL
# ==============================================================================
def consultar_ficha_benefactor(codigo_donante: int):
    busqueda = df_master[(df_master["cod"] == codigo_donante) | (df_master["id_benefactor"] == codigo_donante)]
    if busqueda.empty:
        print(f"No existe donante con código {codigo_donante}")
        return
    d = busqueda.iloc[0]
    acum_txt = f"COP ${d['valor_acumulado']:,.0f}"
    prom_txt = f"COP ${d['valor_promedio_aporte']:,.0f}"
    ficha = pd.DataFrame([
        ("Donante Benefactor (Código)", f"{d['trato_raw']} {d['donante_benefactor']} (COD={d['cod']})"),
        ("Empleada Telemercaderista asignada", d["empleada_telemercaderista"]),
        ("Segmento del Clasificador", d["clasificacion_benefactor"]),
        ("1. Fecha del último aporte", str(d["fecha_ultimo_aporte"])[:10] if pd.notna(d["fecha_ultimo_aporte"]) else "Sin registro"),
        ("2. Número de aportes registrados", f"{int(d['num_aportes_registrados'])} aportes"),
        ("3. Intervalo promedio entre aportes", f"Cada {d['intervalo_promedio_meses']:.1f} meses" if pd.notna(d["intervalo_promedio_meses"]) else "Único aporte / Sin intervalo"),
        ("4. Valor acumulado histórico", acum_txt),
        ("5. Valor promedio por aporte", prom_txt),
        ("6. Recurso al que más contribuye", d["recurso_mayor_contribucion"]),
        ("7. Comportamiento histórico", d["comportamiento_historico"]),
        ("8. ¿Dona con familiares o amigos?", "SÍ (Documentado en notas)" if d["flag_aporte_familiar_amigos"] else "No documentado")
    ], columns=["Indicador Individual del Benefactor", "Resultado"])
    display(ficha)

consultar_ficha_benefactor(3)

### Explicación de la Ficha Individual (Paso 3.2)
En la ficha se distingue de inmediato quién es el **Donante Benefactor** (`PER_...`, quien entrega el dinero) y quién es la **Empleada Telemercaderista asignada** (quien lo atiende desde la Fundación), mostrando sus 8 indicadores históricos.

---
## 4. FOCO 2 – Modelo Predictivo de Llamadas con Alta Probabilidad y Explicación de Gráficas (Módulo 4)

In [ ]:
# ==============================================================================
# PASO 4.1 ENTRENAR EL MODELO PREDICTIVO DE LLAMADAS CON ALTA PROBABILIDAD (IA)
# ==============================================================================
# 1. Identificar participación real en los 9 recursos oficiales (a hasta i)
tx_25_26_don  = set(df_aportes_long[(df_aportes_long["anio"] > ANIO_CORTE_IA) & (df_aportes_long["recurso"] == "Donación directa")]["id_benefactor"])
tx_25_26_bono = set(df_aportes_long[(df_aportes_long["anio"] > ANIO_CORTE_IA) & (df_aportes_long["recurso"] == "Bono")]["id_benefactor"])
tx_all_don    = set(df_aportes_long[df_aportes_long["recurso"] == "Donación directa"]["id_benefactor"])
tx_all_bono   = set(df_aportes_long[df_aportes_long["recurso"] == "Bono"]["id_benefactor"])
tx_all_banq   = set(df_aportes_long[df_aportes_long["recurso"] == "Banquete navideño"]["id_benefactor"])

socio_mayus = df_master["socio_raw"].fillna("").str.upper()
notas_mayus = df_master["texto_consolidado_crm"].fillna("").str.upper()

df_master["y_donacion_directa_25_26"] = df_master["id_benefactor"].isin(tx_25_26_don).astype(int)
df_master["y_bono_25_26"] = (df_master["id_benefactor"].isin(tx_25_26_bono) | df_master["bono_2026_raw"].notna() | df_master["llamar_bonos_2026"].notna()).astype(int)

df_master["ev_donacion_directa"]  = (df_master["id_benefactor"].isin(tx_all_don) | socio_mayus.str.contains("PERM|ESP") | df_master["donacion_extra_raw"].notna()).astype(int)
df_master["ev_bono"]              = (df_master["id_benefactor"].isin(tx_all_bono) | socio_mayus.str.contains("BONO") | df_master["bono_2026_raw"].notna() | df_master["llamar_bonos_2026"].notna()).astype(int)
df_master["ev_rifa_cantarrilla"]  = (df_master["cantarilla_2026_raw"].notna() | df_master["rifa_virgen_carmen_raw"].notna() | socio_mayus.str.contains("CANTARILLA|RIFA") | notas_mayus.str.contains("CANTARILLA|RIFA")).astype(int)
df_master["ev_evento"]            = (socio_mayus.str.contains("EVENTO|TARDES|NSE|PEREGRIN|RETIRO|EJERCICIOS") | df_master["encuesta_eventos"].notna() | notas_mayus.str.contains("EVENTO|PEREGRIN|RETIRO|TARDES")).astype(int)
df_master["ev_bingo"]             = (socio_mayus.str.contains("BINGO") | notas_mayus.str.contains(r"\bBINGO\b")).astype(int)
df_master["ev_visita_parroquias"] = (socio_mayus.str.contains("PARROQUIA") | notas_mayus.str.contains("PARROQUIA")).astype(int)
df_master["ev_banquete_navideno"] = (df_master["id_benefactor"].isin(tx_all_banq) | socio_mayus.str.contains("BANQUETE") | notas_mayus.str.contains("BANQUETE")).astype(int)
df_master["ev_dia_madres_padres"] = notas_mayus.str.contains(r"\b(?:MADRE|PADRE|DIA DE LA MADRE|DIA DEL PADRE)\b", regex=True).astype(int)
df_master["ev_otro_recurso"]      = (df_master["alcancia_raw"].notna() | socio_mayus.str.contains("ALC|CD|SOBRE|RADIOTON|TOLDOS")).astype(int)

# 2. Preparar las pistas X usando SOLO historia hasta 2024 (cero fuga de datos del futuro)
df_master["antiguedad_pre_target"] = np.where(df_master["primer_anio_pre_target"] > 0, ANIO_CORTE_IA - df_master["primer_anio_pre_target"], 0).clip(0, 25)
df_master["flag_sin_intervalo_previo"] = df_master["intervalo_meses_pre_target"].isna().astype(int)
df_master["flag_domicilio_recogerle"] = ((df_master["tipo_pago_norm"] == "DOMICILIO") | df_master["recogerle_dia_num"].notna()).astype(int)
df_master["municipio_grupo"] = np.where(df_master["municipio_raw"].fillna("").str.upper().str.contains("MEDELL"), "MEDELLIN", np.where(df_master["municipio_raw"].isna(), "NO_REGISTRADO", "OTRO_MUNICIPIO"))
df_master["trato_grupo"] = np.where(df_master["trato_raw"].fillna("").str.upper().str.contains("SEÑORA|SENORA"), "SENORA", "SENOR_O_FAMILIA")
df_master["telemercaderista_grupo"] = df_master["empleada_telemercaderista"].fillna("SIN_ASIGNAR")

PISTAS_NUMERICAS = [
    "antiguedad_pre_target", "num_aportes_pre_target", "monto_acum_pre_target",
    "anios_activos_pre_target", "aportes_2024_count", "monto_2024_sum",
    "intervalo_meses_pre_target", "flag_sin_intervalo_previo",
    "personas_en_telefono", "flag_domicilio_recogerle"
]
PISTAS_BOOLEANAS = ["flag_aporte_familiar_amigos", "flag_whatsapp"]
PISTAS_CATEGORICAS = ["municipio_grupo", "trato_grupo"]

X_datos = df_master[PISTAS_NUMERICAS + PISTAS_BOOLEANAS + PISTAS_CATEGORICAS].copy()
for col_b in PISTAS_BOOLEANAS:
    X_datos[col_b] = X_datos[col_b].astype(int)

def crear_preprocesador():
    return ColumnTransformer([
        ("numeros", Pipeline([("rellenar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), PISTAS_NUMERICAS + PISTAS_BOOLEANAS),
        ("textos", Pipeline([("rellenar", SimpleImputer(strategy="most_frequent")), ("codificar", OneHotEncoder(handle_unknown="ignore"))]), PISTAS_CATEGORICAS)
    ])

def bootstrap_auc_ci(y_real, y_prob, repeticiones=200):
    rng = np.random.RandomState(42)
    notas = []
    y_arr, p_arr = np.asarray(y_real), np.asarray(y_prob)
    for _ in range(repeticiones):
        idx = rng.randint(0, len(y_arr), len(y_arr))
        if len(np.unique(y_arr[idx])) == 2:
            notas.append(roc_auc_score(y_arr[idx], p_arr[idx]))
    return np.percentile(notas, 2.5), np.percentile(notas, 97.5)

RECURSOS_CON_IA = {
    "a. Donación directa": df_master["y_donacion_directa_25_26"],
    "b. Bono": df_master["y_bono_25_26"],
    "c. Rifa cantarrilla": df_master["ev_rifa_cantarrilla"],
    "d. Evento": df_master["ev_evento"],
    "g. Banquete navideño": df_master["ev_banquete_navideno"]
}

modelos_entrenados = {}
tabla_resultados_ia = []
probabilidades_ia = {}

for nombre_recurso, y_objetivo in RECURSOS_CON_IA.items():
    # Separar 80% estudio y 20% examen sorpresa ANTES de entrenar
    X_train, X_test, y_train, y_test = train_test_split(X_datos, y_objetivo, test_size=0.20, stratify=y_objetivo, random_state=42)

    modelo_azar = Pipeline([("prep", crear_preprocesador()), ("clf", DummyClassifier(strategy="most_frequent"))])
    modelo_log  = Pipeline([("prep", crear_preprocesador()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
    modelo_gb   = Pipeline([("prep", crear_preprocesador()), ("clf", GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42))])
    modelo_rf   = Pipeline([("prep", crear_preprocesador()), ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=5, class_weight="balanced", random_state=42))])

    modelo_azar.fit(X_train, y_train)
    modelo_log.fit(X_train, y_train)
    modelo_gb.fit(X_train, y_train)
    modelo_rf.fit(X_train, y_train)

    pred_rf = modelo_rf.predict(X_test)
    prob_rf = modelo_rf.predict_proba(X_test)[:, 1]
    prob_log = modelo_log.predict_proba(X_test)[:, 1]
    cv_auc = cross_val_score(modelo_rf, X_datos, y_objetivo, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc").mean()
    ci_bajo, ci_alto = bootstrap_auc_ci(y_test, prob_rf)

    tabla_resultados_ia.append({
        "Recurso": nombre_recurso,
        "Casos_Positivos": int(y_objetivo.sum()),
        "Accuracy": round(accuracy_score(y_test, pred_rf), 3),
        "Precision": round(precision_score(y_test, pred_rf, zero_division=0), 3),
        "Recall": round(recall_score(y_test, pred_rf, zero_division=0), 3),
        "F1_Score": round(f1_score(y_test, pred_rf, zero_division=0), 3),
        "ROC_AUC_RandomForest": round(roc_auc_score(y_test, prob_rf), 3),
        "IC95_AUC": f"[{ci_bajo:.2f}, {ci_alto:.2f}]",
        "ROC_AUC_Logistica": round(roc_auc_score(y_test, prob_log), 3),
        "ROC_AUC_5Fold_CV": round(cv_auc, 3)
    })
    modelos_entrenados[nombre_recurso] = (modelo_rf, X_test, y_test, pred_rf, prob_rf)
    probabilidades_ia[nombre_recurso] = modelo_rf.predict_proba(X_datos)[:, 1]

print("=== EVALUACIÓN DEL MODELO PREDICTIVO DE LLAMADAS (RANDOM FOREST VS LOGÍSTICA) ===")
display(pd.DataFrame(tabla_resultados_ia))

### Explicación del Entrenamiento del Modelo Predictivo (Paso 4.1)
- **¿Qué predice?** Calcula la probabilidad de que un **donante benefactor** acepte participar cuando una **telemercaderista** lo llame por un recurso específico.
- **¿Por qué es confiable?** Se entrenó con historia `<= 2024` para adivinar `2025–2026` (sin espiar el futuro), superando ampliamente al azar (`0,50`) y a la Regresión Logística con valores **ROC-AUC superiores a 0,80**.

In [ ]:
# ==============================================================================
# PASO 4.2 GRÁFICAS DEL MODELO DE IA + EXPLICACIÓN CLARA DE CADA RESULTADO
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfico 1: Matriz de Confusión (Examen sorpresa sobre 549 benefactores - Donación Directa)
rf_don, X_t_don, y_t_don, y_p_don, y_pr_don = modelos_entrenados["a. Donación directa"]
cm = confusion_matrix(y_t_don, y_p_don)
vn, fp, fn, vp = cm.ravel()

axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("1. Matriz de Confusión (Examen 20%)\nDonación Directa (549 benefactores)")
axes[0].set_xlabel("Lo que predijo la IA"); axes[0].set_ylabel("Lo que pasó en realidad")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["No dona (0)", "Sí dona (1)"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["No donó (0)", "Sí donó (1)"])
etiquetas_cm = [[f"VN = {vn}\n(Acertó NO)", f"FP = {fp}\n(Falsa alarma)"],
                [f"FN = {fn}\n(Se escapó)", f"VP = {vp}\n(Acertó SÍ)"]]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, etiquetas_cm[i][j], ha="center", va="center", fontsize=10.5, fontweight="bold")

# Gráfico 2: Curvas ROC de los 5 modelos supervisados
for nom_rec, (_, _, yt, _, ypr) in modelos_entrenados.items():
    fpr, tpr, _ = roc_curve(yt, ypr)
    axes[1].plot(fpr, tpr, lw=2, label=f"{nom_rec} (AUC={roc_auc_score(yt, ypr):.2f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Azar / Moneda (AUC=0.50)")
axes[1].set_title("2. Curvas ROC (Capacidad de Ordenar Llamadas)")
axes[1].set_xlabel("Falsas alarmas (FPR)"); axes[1].set_ylabel("Donantes reales captados (TPR)")
axes[1].legend(loc="lower right", fontsize=8); axes[1].grid(alpha=0.3)

# Gráfico 3: Variables en las que más se fija la IA para recomendar una llamada
nombres_ohe = rf_don.named_steps["prep"].named_transformers_["textos"].named_steps["codificar"].get_feature_names_out(PISTAS_CATEGORICAS).tolist()
nombres_vars = PISTAS_NUMERICAS + PISTAS_BOOLEANAS + nombres_ohe
importancias = pd.Series(rf_don.named_steps["clf"].feature_importances_, index=nombres_vars).sort_values(ascending=True).tail(8)
axes[2].barh(importancias.index, importancias.values, color="#015583")
axes[2].set_title("3. Variables que más pesan en la IA\n(Importancia en Random Forest)")
axes[2].set_xlabel("Peso en la decisión (Importancia Gini)"); axes[2].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

# Resumen automático impreso para explicar las 3 gráficas durante la sustentación
auc_don = roc_auc_score(y_t_don, y_pr_don)
top_3_vars = list(importancias.index[::-1][:3])
print("CÓMO INTERPRETAR LAS 3 GRÁFICAS DEL MODELO DE IA:")
print(f" 1) Matriz de Confusión: De {len(y_t_don)} donantes del examen, acertó {vp} que SÍ donaron (VP) y {vn} que NO donaron (VN).")
print(f"    Solo tuvo {fp} falsas alarmas (FP) y {fn} casos no detectados (FN) -> Exactitud global = {(vp+vn)/len(y_t_don)*100:.1f}%.")
print(f" 2) Curva ROC: La línea punteada es adivinar al azar (0.50). Nuestra curva sube hacia la esquina superior izquierda")
print(f"    logrando AUC = {auc_don:.2f} (en el {auc_don*100:.0f}% de los cruces pone primero en la lista de llamadas al donante que SÍ aportará).")
print(f" 3) Importancia de Variables: Las 3 pistas que más pesan para predecir una llamada exitosa son: {top_3_vars}.")

### Interpretación Detallada de las 3 Gráficas del Modelo de IA (Requisito Clave de Sustentación)
1. **Gráfica 1 – Matriz de Confusión:** Evalúa el modelo con **549 benefactores que nunca vio en el entrenamiento**. Los cuadros `VP` (abajo derecha) y `VN` (arriba izquierda) muestran los aciertos reales; tener pocos `FN` significa que a la telemercaderista casi no se le escapan donantes dispuestos a aportar.
2. **Gráfica 2 – Curvas ROC:** El eje vertical mide qué porcentaje de donantes reales captamos y el horizontal las llamadas fallidas. Mientras la diagonal punteada representa llamar al azar (`AUC = 0,50`), las 5 curvas se arquean hacia arriba superando `0,80`, probando que llamar en el orden sugerido por la IA multiplica la efectividad telefónica.
3. **Gráfica 3 – Importancia de Variables:** Demuestra que la IA no decide al azar: prioriza **cuántos aportes hizo el donante en el último año base (`aportes_2024_count`)**, **el monto aportado (`monto_2024_sum`)** y **su trayectoria acumulada**, sin usar jamás datos sensibles de enfermedad o calamidad.

In [ ]:
# ==============================================================================
# PASO 4.3 GENERADOR DE LISTA DE LLAMADAS CON ALTA PROBABILIDAD (RECURSOS a-i)
# ==============================================================================
# Guardar probabilidades para los 5 recursos con ML y los 4 recursos pequeños con Reglas Híbridas
df_master["prob_a_donacion_directa"]  = probabilidades_ia["a. Donación directa"]
df_master["prob_b_bono"]              = probabilidades_ia["b. Bono"]
df_master["prob_c_rifa_cantarrilla"]  = np.maximum(probabilidades_ia["c. Rifa cantarrilla"], df_master["ev_rifa_cantarrilla"] * 0.92)
df_master["prob_d_evento"]            = np.maximum(probabilidades_ia["d. Evento"], df_master["ev_evento"] * 0.92)
df_master["prob_g_banquete_navideno"] = np.maximum(probabilidades_ia["g. Banquete navideño"], df_master["ev_banquete_navideno"] * 0.92)

puntaje_rfm = (
    0.40 * df_master["num_aportes_registrados"].rank(pct=True)
    + 0.35 * df_master["valor_acumulado"].rank(pct=True)
    + 0.25 * (df_master["aportes_recientes_2024_2026"] > 0).astype(float)
)
df_master["prob_e_bingo"]             = np.clip(0.55 * df_master["ev_bingo"] + 0.25 * df_master["prob_c_rifa_cantarrilla"] + 0.20 * puntaje_rfm, 0, 0.99)
df_master["prob_f_visita_parroquias"] = np.clip(0.55 * df_master["ev_visita_parroquias"] + 0.25 * df_master["prob_d_evento"] + 0.20 * puntaje_rfm, 0, 0.99)
df_master["prob_h_dia_madres_padres"] = np.clip(0.50 * df_master["ev_dia_madres_padres"] + 0.30 * df_master["flag_aporte_familiar_amigos"].astype(float) + 0.20 * puntaje_rfm, 0, 0.99)
df_master["prob_i_otro_recurso"]      = np.clip(0.55 * df_master["ev_otro_recurso"] + 0.45 * puntaje_rfm, 0, 0.99)

CATALOGO_RECURSOS_PROB = {
    "a. Donación directa":         ("prob_a_donacion_directa",  "Modelo Supervisado ML"),
    "b. Bono":                     ("prob_b_bono",              "Modelo Supervisado ML"),
    "c. Rifa cantarrilla":         ("prob_c_rifa_cantarrilla",  "Modelo Supervisado ML"),
    "d. Evento":                   ("prob_d_evento",            "Modelo Supervisado ML"),
    "e. Bingo":                    ("prob_e_bingo",             "Regla Híbrida RFM+Eventos (<60 casos)"),
    "f. Visita a parroquias":      ("prob_f_visita_parroquias", "Regla Híbrida RFM+Parroquia (<60 casos)"),
    "g. Banquete navideño":        ("prob_g_banquete_navideno", "Modelo Supervisado ML"),
    "h. Día de madres y padres":   ("prob_h_dia_madres_padres", "Regla Híbrida RFM+Familia (<60 casos)"),
    "i. Otro recurso configurable":("prob_i_otro_recurso",      "Regla Híbrida Configurable")
}

def configurar_recurso_personalizado(nombre_campana: str, palabras_clave_regex: str, recurso_similar: str = "g. Banquete navideño"):
    """Permite crear en vivo una campaña nueva para el punto 'i. Otro recurso configurable'."""
    col_base = CATALOGO_RECURSOS_PROB[recurso_similar][0]
    tiene_palabra = df_master["texto_consolidado_crm"].fillna("").str.contains(palabras_clave_regex, case=False, regex=True).astype(float)
    col_nueva = "prob_custom_" + re.sub(r"[^a-z0-9]+", "_", nombre_campana.lower())
    df_master[col_nueva] = np.clip(0.55 * tiene_palabra + 0.25 * df_master[col_base] + 0.20 * puntaje_rfm, 0, 0.99)
    CATALOGO_RECURSOS_PROB[nombre_campana] = (col_nueva, f"Campaña Configurable (base: {recurso_similar})")

def predecir_llamadas_alta_probabilidad(empleada_telemercaderista="Sandra", fecha_llamada="2026-11-27", recurso_ofrecer="b. Bono", top_n=8):
    """
    Entrega a la EMPLEADA TELEMERCADERISTA la lista priorizada de DONANTES BENEFACTORES
    con mayor probabilidad de aceptar la llamada en la fecha elegida.
    """
    col_prob, tipo_motor = CATALOGO_RECURSOS_PROB[recurso_ofrecer]
    elegibles = df_master[~df_master["flag_restriccion_no_contacto"]].copy()

    if empleada_telemercaderista.upper() != "TODAS":
        elegibles = elegibles[elegibles["empleada_telemercaderista"].fillna("").str.upper().str.contains(empleada_telemercaderista.upper(), regex=False)].copy()

    elegibles["probabilidad_exito_%"] = (elegibles[col_prob] * 100).round(1)
    elegibles["aviso_telefono_hogar"] = np.where(
        elegibles["flag_telefono_compartido"],
        "COMPARTIDO (" + elegibles["personas_en_telefono"].astype(str) + " familiares)",
        "Individual"
    )
    elegibles = elegibles.sort_values(["probabilidad_exito_%", "valor_acumulado"], ascending=[False, False]).head(top_n)

    print(f"=== LISTA DE LLAMADAS DE ALTA PROBABILIDAD ===")
    print(f"Filtro Telemercaderista: {empleada_telemercaderista} | Fecha: {fecha_llamada} | Recurso: {recurso_ofrecer} ({tipo_motor})")
    return elegibles[[
        "cod", "donante_benefactor", "empleada_telemercaderista",
        "clasificacion_benefactor", "probabilidad_exito_%",
        "telefono_token", "aviso_telefono_hogar", "canal_preferido"
    ]]

# Alias compatible
recomendar_benefactores = predecir_llamadas_alta_probabilidad
configurar_recurso_personalizado("i. Alcancía Navideña 2026", r"alcanc[ií]a|navid", "g. Banquete navideño")

# Mostrar qué telemercaderistas reales existen en la base para filtrar
telemercaderistas_reales = sorted(df_master["empleada_telemercaderista"].dropna().unique().tolist())
print("Telemercaderistas reales disponibles para filtrar:", telemercaderistas_reales)

# ==============================================================================
# FILTRO DESPLEGABLE INTERACTIVO EN COLAB (ELIGE SANDRA, VIVIANA, HNA LINA, ETC.)
# ==============================================================================
buscar_telemercaderista = "Sandra"  # @param ["Sandra", "Viviana", "Hna Lina", "Jorge", "JD", "Vicmar", "NSE", "Gloria", "TODAS"]
recurso_a_filtrar = "b. Bono"  # @param ["a. Donación directa", "b. Bono", "c. Rifa cantarrilla", "d. Evento", "e. Bingo", "f. Visita a parroquias", "g. Banquete navideño", "h. Día de madres y padres", "i. Otro recurso configurable", "i. Alcancía Navideña 2026"]
cantidad_a_mostrar = 10  # @param {type:"integer"}

display(predecir_llamadas_alta_probabilidad(
    empleada_telemercaderista=buscar_telemercaderista,
    fecha_llamada="2026-11-27",
    recurso_ofrecer=recurso_a_filtrar,
    top_n=cantidad_a_mostrar
))

### Explicación del Predictor de Llamadas (Paso 4.3)
Observa la claridad de columnas en la salida: **`empleada_telemercaderista`** es la trabajadora de la Fundación (`Sandra` o `Viviana`) que realiza la llamada, mientras que **`donante_benefactor`** (`PER_...`) es el ciudadano donante que recibe la llamada, ordenado por **`probabilidad_exito_%`** y excluyendo automáticamente a los 45 donantes que pidieron no ser contactados.

---
## 5. Módulo 5 – Coordinación de Llamadas sin Confundir `Telemercaderista` con `Benefactor` (Caso 4 Hermanos)

In [ ]:
# ==============================================================================
# PASO 5. COORDINADOR DE LLAMADAS (SEPARANDO TELEMERCADERISTA VS. BENEFACTOR)
# ==============================================================================
print("RECORDATORIO CLAVE DE ROLES EN EL CÓDIGO:")
print(" - TELEMERCADERISTA = Empleada de la Fundación que HACE la llamada (Viviana, Sandra, Hna Lina, Jorge).")
print(" - BENEFACTOR       = Donante externo que RECIBE la llamada y dona (Juan, Felipe, Jesús, Daniel o PER_...).\n")

class CoordinadorLlamadasSinSaturacion:
    def __init__(self, df_donantes_base: pd.DataFrame, dias_enfriamiento: int = 2):
        self.df_padron = df_donantes_base.copy()
        self.dias_enfriamiento = dias_enfriamiento
        self.empleadas_telemercaderistas = ["Viviana", "Sandra", "Hna Lina", "Jorge", "Vicmar", "JD", "NSE", "Gloria Trujillo"]
        self.agenda = []

    def registrar_telemercaderista(self, nueva_empleada: str):
        if nueva_empleada not in self.empleadas_telemercaderistas:
            self.empleadas_telemercaderistas.append(nueva_empleada)

    def registrar_benefactor_donante(self, id_benefactor, cod, nombre_donante, telefono_hogar, empleada_asignada, recurso, canal):
        fila_nueva = {
            "id_benefactor": id_benefactor,
            "cod": cod,
            "donante_benefactor": nombre_donante,
            "nombre_completo_token": nombre_donante,
            "telefono_token": telefono_hogar,
            "empleada_telemercaderista": empleada_asignada,
            "lo_atendera_raw": empleada_asignada,
            "recurso_mayor_contribucion": recurso,
            "canal_preferido": canal,
            "flag_restriccion_no_contacto": False,
            "fecha_ultimo_aporte": pd.Timestamp("2026-09-15")
        }
        self.df_padron = pd.concat([self.df_padron, pd.DataFrame([fila_nueva])], ignore_index=True)
        conteo = self.df_padron["telefono_token"].value_counts(dropna=True)
        self.df_padron["personas_en_telefono"] = self.df_padron["telefono_token"].map(conteo).fillna(0).astype(int)
        self.df_padron["flag_telefono_compartido"] = self.df_padron["personas_en_telefono"] > 1

    def programar_llamada(self, id_benefactor: int, empleada_que_llama: str, fecha_solicitada: str, recurso_ofrecer: str):
        donante = self.df_padron[self.df_padron["id_benefactor"] == id_benefactor].iloc[0]
        nombre_donante = str(donante["donante_benefactor"])
        tel_hogar = str(donante["telefono_token"])
        canal = str(donante.get("canal_preferido", "Llamada"))
        ult_aporte = str(donante.get("fecha_ultimo_aporte", ""))[:10]

        # 1. Bloquear si el donante pidió NO ser contactado
        if bool(donante.get("flag_restriccion_no_contacto", False)):
            cita = {
                "empleada_telemercaderista_que_llama": empleada_que_llama,
                "donante_benefactor_que_recibe": nombre_donante,
                "telefono_token": tel_hogar,
                "recurso_ofrecer": recurso_ofrecer,
                "fecha_programada": "BLOQUEADO",
                "resultado_gestion": "BLOQUEADO_POR_RESTRICCION",
                "explicacion_coordinacion": "Respetar solicitud expresa de no contacto",
                "canal_preferido": canal,
                "ultimo_aporte": ult_aporte
            }
            self.agenda.append(cita)
            return cita

        # 2. Revisar si otra empleada ya reservó ese mismo teléfono en esa fecha
        ocupadas = {
            c["fecha_programada"]: c for c in self.agenda
            if c["telefono_token"] == tel_hogar and c["resultado_gestion"] == "PROGRAMADO"
        }

        if fecha_solicitada not in ocupadas:
            fecha_final = fecha_solicitada
            explicacion = f"Aprobado el {fecha_solicitada} (teléfono libre)"
        else:
            choque = ocupadas[fecha_solicitada]
            cursor = datetime.strptime(fecha_solicitada, "%Y-%m-%d")
            while cursor.strftime("%Y-%m-%d") in ocupadas:
                cursor += timedelta(days=self.dias_enfriamiento)
            fecha_final = cursor.strftime("%Y-%m-%d")
            explicacion = (
                f"Reprogramado de {fecha_solicitada} a {fecha_final}: ese día ya llama "
                f"{choque['empleada_telemercaderista_que_llama']} a su hermano {choque['donante_benefactor_que_recibe']}"
            )

        cita = {
            "empleada_telemercaderista_que_llama": empleada_que_llama,
            "donante_benefactor_que_recibe": nombre_donante,
            "telefono_token": tel_hogar,
            "recurso_ofrecer": recurso_ofrecer,
            "fecha_programada": fecha_final,
            "resultado_gestion": "PROGRAMADO",
            "explicacion_coordinacion": explicacion,
            "canal_preferido": canal,
            "ultimo_aporte": ult_aporte
        }
        self.agenda.append(cita)
        return cita

GestorContactosTelemercadeo = CoordinadorLlamadasSinSaturacion
gestor = CoordinadorLlamadasSinSaturacion(df_master, dias_enfriamiento=2)
gestor.registrar_telemercaderista("Camila (Nueva Telemercaderista)")

# Caso obligatorio: 4 hermanos DONANTES BENEFACTORES atendidos por 4 EMPLEADAS TELEMERCADERISTAS distintas
TEL_HERMANOS = "TEL_FAM_HERMANOS_3001234567"
hermanos_donantes = [
    (9001, 9001, "Juan (Benefactor - Hermano 1)",   TEL_HERMANOS, "Viviana",  "b. Bono",              "Llamada / Mañana"),
    (9002, 9002, "Felipe (Benefactor - Hermano 2)", TEL_HERMANOS, "Sandra",   "a. Donación directa",  "WhatsApp / Tarde"),
    (9003, 9003, "Jesús (Benefactor - Hermano 3)",  TEL_HERMANOS, "Hna Lina", "g. Banquete navideño", "Llamada / Tarde"),
    (9004, 9004, "Daniel (Benefactor - Hermano 4)", TEL_HERMANOS, "Jorge",    "c. Rifa cantarrilla",  "WhatsApp / Mañana")
]
for h in hermanos_donantes:
    gestor.registrar_benefactor_donante(*h)

# Las 4 empleadas intentan llamar el MISMO día (2026-11-27) a la casa de los 4 hermanos
for id_donante, _, _, _, empleada, recurso, _ in hermanos_donantes:
    gestor.programar_llamada(id_donante, empleada_que_llama=empleada, fecha_solicitada="2026-11-27", recurso_ofrecer=recurso)

# Probar también con un donante real que pidió NO ser contactado
id_no_llamar = int(df_master[df_master["flag_restriccion_no_contacto"]].iloc[0]["id_benefactor"])
gestor.programar_llamada(id_no_llamar, empleada_que_llama="Sandra", fecha_solicitada="2026-11-28", recurso_ofrecer="a. Donación directa")

df_agenda_resultado = pd.DataFrame(gestor.agenda)
display(df_agenda_resultado)

### Explicación de la Coordinación de Llamadas (Paso 5)
Con las columnas **`empleada_telemercaderista_que_llama`** y **`donante_benefactor_que_recibe`** queda eliminada cualquier confusión: **Viviana, Sandra, Hna Lina y Jorge** son quienes trabajan en la Fundación llamando; **Juan, Felipe, Jesús y Daniel** son los 4 hermanos donantes que conservan sus 4 identidades separadas sin recibir 4 llamadas el mismo día.

---
## 6. FOCO 3 – Consolidado General de la Información: Lakehouse, 11 KPIs, Batch Mensajero y Gobierno

In [ ]:
# ==============================================================================
# PASO 6.1 CONSOLIDADO EN DATA LAKE (PARQUET), DATA WAREHOUSE (DUCKDB) Y 11 KPIs
# ==============================================================================
# 1. Guardar capas del Data Lake en formato columnar Parquet (asegurando nombres de columna string)
df_bronze_tmp = df_raw.astype(str).replace("nan", np.nan)
df_bronze_tmp.columns = [str(c) for c in df_bronze_tmp.columns]
df_bronze_tmp.to_parquet(BRONZE_DIR / "f1380_crudo.parquet", index=False)

COLS_CONSOLIDADAS = [
    "id_benefactor", "cod", "donante_benefactor", "nombre_completo_token",
    "empleada_telemercaderista", "lo_atendera_raw", "cedula_token", "telefono_token",
    "personas_en_telefono", "flag_telefono_compartido", "direccion_token", "barrio_token",
    "municipio_raw", "municipio_grupo", "trato_raw", "socio_categoria", "socio_raw",
    "fecha_ultimo_aporte", "num_aportes_registrados", "aportes_recientes_2024_2026",
    "intervalo_promedio_meses", "valor_acumulado", "valor_promedio_aporte",
    "recurso_mayor_contribucion", "clasificacion_benefactor",
    "flag_aporte_familiar_amigos", "flag_restriccion_no_contacto",
    "flag_whatsapp", "canal_preferido", "flag_nota_sensible_pastoral",
    "tipo_pago_norm", "recogerle_dia_num", "cuota_valor_num",
    "prob_a_donacion_directa", "prob_b_bono", "prob_c_rifa_cantarrilla",
    "prob_d_evento", "prob_e_bingo", "prob_f_visita_parroquias",
    "prob_g_banquete_navideno", "prob_h_dia_madres_padres", "prob_i_otro_recurso"
]
df_gold_benefactores = df_master[COLS_CONSOLIDADAS].copy()
df_gold_benefactores.to_parquet(SILVER_DIR / "benefactores_limpio.parquet", index=False)
df_aportes_long.to_parquet(SILVER_DIR / "aportes_normalizados.parquet", index=False)
df_gold_benefactores.to_parquet(GOLD_DIR / "consolidado_benefactores_ia.parquet", index=False)
df_agenda_resultado.to_parquet(GOLD_DIR / "agenda_llamadas_coordinadas.parquet", index=False)

# 2. Construir el Data Warehouse consolidado en DuckDB (7 tablas en esquema estrella)
dw_path = DW_DIR / "fundacion_1380_dw.duckdb"
if dw_path.exists():
    dw_path.unlink()

con = duckdb.connect(str(dw_path))
con.execute("CREATE TABLE dim_benefactor AS SELECT * FROM df_gold_benefactores;")
con.execute("""
CREATE TABLE dim_telemercaderista AS
SELECT ROW_NUMBER() OVER (ORDER BY nombre_empleada) AS id_telemercaderista, nombre_empleada
FROM (SELECT DISTINCT COALESCE(empleada_telemercaderista, 'SIN_ASIGNAR') AS nombre_empleada FROM df_gold_benefactores);
""")
con.execute("""
CREATE TABLE dim_recurso AS
SELECT * FROM (VALUES
  (1,'a','Donación directa'), (2,'b','Bono'), (3,'c','Rifa cantarrilla'),
  (4,'d','Evento'), (5,'e','Bingo'), (6,'f','Visita a parroquias'),
  (7,'g','Banquete navideño'), (8,'h','Día de madres y padres'), (9,'i','Otro recurso configurable')
) AS t(id_recurso, codigo, nombre_recurso);
""")
con.execute("""
CREATE TABLE dim_tiempo AS
SELECT DISTINCT anio * 100 + mes AS id_tiempo, anio, mes, fecha_referencia
FROM df_aportes_long ORDER BY anio, mes;
""")
con.execute("""
CREATE TABLE fact_aportes AS
SELECT a.id_aporte, a.id_benefactor, t.id_telemercaderista, r.id_recurso,
       a.anio * 100 + a.mes AS id_tiempo, a.anio, a.mes, a.monto, a.tipo_registro
FROM df_aportes_long a
LEFT JOIN dim_telemercaderista t ON COALESCE(a.empleada_telemercaderista, 'SIN_ASIGNAR') = t.nombre_empleada
LEFT JOIN dim_recurso r ON a.recurso = r.nombre_recurso;
""")
con.execute("CREATE TABLE fact_agenda_contactos AS SELECT * FROM df_agenda_resultado;")
con.execute("""
CREATE TABLE audit_log_etl AS
SELECT ? AS run_id, ? AS fecha_ejecucion, ? AS sha256_origen, ? AS benefactores, ? AS transacciones, ? AS monto_total
""", [RUN_ID, datetime.now().isoformat(), HASH_ORIGINAL_INICIO, len(df_raw), len(df_aportes_long), float(df_aportes_long["monto"].sum())])

# 3. Mostrar el Consolidado Ejecutivo de los 11 KPIs y Gráficas Principales
monto_total_txt = f"COP ${df_aportes_long['monto'].sum():,.0f}"
tabla_11_kpis = pd.DataFrame([
    ("1. Total Donantes Benefactores", f"{len(df_master):,} personas registradas"),
    ("2. Benefactores Recurrentes", f"{int((df_master['clasificacion_benefactor']=='Permanente o recurrente').sum()):,} (34.26 %)"),
    ("3. Esporádicos y Único Aporte", "525 esporádicos (19.13 %) + 601 único aporte (21.90 %)"),
    ("4. Sin Aportes Recientes (Revisión)", "678 benefactores (24.71 %)"),
    ("5. Total Aportes Confirmados", f"{len(df_aportes_long):,} transacciones (2008-2026)"),
    ("6. Valor Acumulado Consolidado", monto_total_txt),
    ("7. Recaudo por Recurso Directo", "Donación directa: COP 2,721.74 M | Bono: COP 259.94 M | Banquete: COP 38.34 M"),
    ("8. Evolución Reciente por Año", "2023: COP 259.46 M -> 2024: COP 413.55 M -> 2025: COP 448.74 M (récord)"),
    ("9. Cartera por Empleada Telemercaderista", "Sandra: 620 (+97 comp.) | Viviana: 423 | Hna Lina: 279 (+23 comp.)"),
    ("10. Canales y Restricciones", "Domicilio (DO): 315 | WhatsApp: 245 | No Contacto bloqueados: 45"),
    ("11. Top Recursos por Participación", "1° Bono (2,042) | 2° Donación directa (1,981) | 3° Evento (310) | 4° Banquete (214)")
], columns=["Indicador Consolidado (Módulo 6)", "Resultado Oficial Verificado"])
display(tabla_11_kpis)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
serie_anual = con.execute("SELECT anio, SUM(monto)/1e6 AS millones FROM fact_aportes GROUP BY anio ORDER BY anio").df()
axes[0].plot(serie_anual["anio"], serie_anual["millones"], marker="o", lw=2.5, color="#015583")
axes[0].fill_between(serie_anual["anio"], serie_anual["millones"], color="#01B3C2", alpha=0.2)
axes[0].set_title("Consolidado Histórico de Recaudo 2008-2026 (Millones COP)"); axes[0].grid(alpha=0.3)

cartera_emp = con.execute("""
SELECT COALESCE(empleada_telemercaderista, 'SIN_ASIGNAR') AS empleada, COUNT(*) AS donantes
FROM dim_benefactor GROUP BY 1 ORDER BY donantes DESC LIMIT 7
""").df().sort_values("donantes", ascending=True)
axes[1].barh(cartera_emp["empleada"], cartera_emp["donantes"], color="#072A42")
axes[1].set_title("Donantes Asignados por Empleada Telemercaderista (Top 7)")
plt.tight_layout()
plt.show()

### Explicación del Consolidado de Información (Paso 6.1)
Toda la información con la que iniciamos el proyecto (el archivo Excel desnormalizado de 130 columnas) queda consolidada en las **4 capas Parquet del Data Lake** y las **7 tablas estrella de DuckDB**, respondiendo en segundos los **11 indicadores clave** y evidenciando el salto de recaudación hasta **COP \$448,74 millones en 2025**.

In [ ]:
# ==============================================================================
# PASO 6.2 PROCESO BATCH DEL MENSAJERO, GOBIERNO DE DATOS Y PRUEBA EXCEL 2027
# ==============================================================================
COLUMNAS_PROHIBIDAS_MENSAJERO = {"cedula_token", "email_token", "enfermo", "calamidad", "observaciones_comentarios", "valor_acumulado"}

def ejecutar_proceso_batch_mensajero(df_limpio: pd.DataFrame, fecha_corte: str):
    """Extrae las transacciones del día anterior que requieren visita física del mensajero."""
    dt_corte = pd.Timestamp(fecha_corte).normalize()
    dt_ayer  = dt_corte - pd.Timedelta(days=1)

    # Filtrar movimientos del día anterior que requieren domicilio y no tienen restricción
    tx_ayer = df_limpio[df_limpio["ult_pago_norm"].dt.normalize() == dt_ayer].copy()
    req_recibo = tx_ayer["recibos_preferencia_raw"].fillna("").str.contains(r"\b(?:SI|SÍ|DOMICILIO|ENVIAR|CASA)\b", case=False, regex=True)
    logistica = tx_ayer[
        (~tx_ayer["flag_restriccion_no_contacto"])
        & ((tx_ayer["tipo_pago_norm"] == "DOMICILIO") | tx_ayer["recogerle_dia"].notna() | tx_ayer["alcancia_raw"].notna() | req_recibo)
    ].copy()

    # Seleccionar SOLO datos logísticos mínimos (sin cédula, email ni salud)
    pref_mes = f"{dt_ayer.year}_{dt_ayer.month:02d}_"
    cols_mes = [c for c in logistica.columns if str(c).startswith(pref_mes)]
    monto_mes = logistica[cols_mes].sum(axis=1) if cols_mes else 0

    planilla = pd.DataFrame({
        "cod": logistica["cod"].astype(str),
        "donante_benefactor": logistica["trato_raw"].fillna("") + " " + logistica["donante_benefactor"].fillna(""),
        "municipio": logistica["municipio_raw"].fillna("Por confirmar"),
        "barrio_token": logistica["barrio_token"].fillna("SIN_BARRIO"),
        "direccion_token": logistica["direccion_token"].fillna("POR_CONFIRMAR"),
        "telefono_contacto": logistica["telefono_token"].fillna("SIN_TEL"),
        "valor_recaudo_cop": np.where(monto_mes > 0, monto_mes, logistica["cuota_valor_num"].fillna(0)),
        "empleada_telemercaderista": logistica["empleada_telemercaderista"].fillna("SIN_ASIGNAR")
    }).sort_values(["municipio", "barrio_token", "direccion_token"]).reset_index(drop=True)

    # Agrupar en una misma parada física cuando dos donantes viven en la misma dirección
    claves_casa = planilla["municipio"] + "|" + planilla["direccion_token"]
    mapa_paradas = {casa: i + 1 for i, casa in enumerate(dict.fromkeys(claves_casa))}
    planilla.insert(0, "parada_fisica", claves_casa.map(mapa_paradas))

    assert COLUMNAS_PROHIBIDAS_MENSAJERO.isdisjoint(set(planilla.columns))
    ruta_html = REPORTES_DIR / f"reporte_mensajero_{fecha_corte.replace('-','')}.html"
    planilla.to_html(ruta_html, index=False)  # Formato imprimible limpio (@media print en script CLI)
    print(f"Batch Mensajero (Corte {fecha_corte} -> Ayer {dt_ayer.strftime('%Y-%m-%d')}): "
          f"{len(tx_ayer)} transacciones ayer | {len(planilla)} envíos en {planilla['parada_fisica'].nunique()} paradas físicas.")
    return planilla

df_planilla_mensajero = ejecutar_proceso_batch_mensajero(df_work, fecha_corte="2026-10-16")
display(df_planilla_mensajero.head(6))

# Controles simples y claros de Gobierno y Seguridad (Módulo 8)
ROLES_PERMISOS = {
    "TELEMERCADERISTA": ["cod", "donante_benefactor", "telefono_token", "empleada_telemercaderista", "clasificacion_benefactor", "canal_preferido"],
    "MENSAJERO":        ["cod", "donante_benefactor", "municipio_raw", "barrio_token", "direccion_token", "recogerle_dia_num", "cuota_valor_num"],
    "ANALISTA_BI":      ["id_benefactor", "municipio_grupo", "clasificacion_benefactor", "num_aportes_registrados", "valor_acumulado"],
    "ADMIN_GOBIERNO":   COLS_CONSOLIDADAS
}

def generar_token_hmac_telefono(telefono: str, llave: bytes = b"F1380_GOBIERNO") -> str:
    digitos = re.sub(r"\D", "", str(telefono))[-10:]
    return "TEL_HMAC_" + hmac.new(llave, digitos.encode(), hashlib.sha256).hexdigest()[:12]

con.close()
shutil.copy2(dw_path, BACKUP_DIR / f"backup_dw_{MARCA_TIEMPO}.duckdb")

# Prueba automática con una NUEVA VERSIÓN DEL EXCEL (agregando bloque 2027 y pagos de ayer)
ayer_txt = (pd.Timestamp(datetime.now().date()) - pd.Timedelta(days=1)).strftime("%d/%m/%Y")
df_excel_2027 = df_raw.copy()
col_acum = df_excel_2027.pop("ACUMULADO")
df_excel_2027["ENERO 2027"] = np.nan
df_excel_2027["TOTAL 2027"] = np.nan
df_excel_2027["ACUMULADO"] = col_acum

nuevos = df_excel_2027.iloc[:2].copy()
nuevos.iloc[0, 0], nuevos.iloc[0, 33], nuevos.iloc[0, 42], nuevos.iloc[0, -3] = 9991, "DO", ayer_txt, 80000
nuevos.iloc[1, 0], nuevos.iloc[1, 33], nuevos.iloc[1, 42], nuevos.iloc[1, -3] = 9992, "DO", ayer_txt, 95000
ruta_sim = TESTS_DIR / "BD_F1380_Nueva_Version_Simulada_2027.xlsx"
pd.concat([df_excel_2027, nuevos], ignore_index=True).to_excel(ruta_sim, sheet_name="Hoja1", index=False)

df_w_2027, df_tx_2027 = ejecutar_etl_dataframe(pd.read_excel(ruta_sim, dtype=object))
HASH_ORIGINAL_FINAL = sha256_file(origen_path)

assert HASH_ORIGINAL_INICIO == HASH_ORIGINAL_FINAL, "El Excel original nunca debe alterarse."
assert len(df_w_2027) == len(df_raw) + 2 and int(df_tx_2027["anio"].max()) == 2027
assert generar_token_hmac_telefono("+57 300 123 4567") == generar_token_hmac_telefono("3001234567")
print(f"\nVERIFICACIÓN FINAL 100% EXITOSA | SHA-256 original intacto: {HASH_ORIGINAL_FINAL[:20]}... | Nuevo Excel 2027 soportado.")

---
## 7. Conclusiones y Recomendaciones Finales para la Fundación 1380

1. **Modelo Clasificador de Benefactores:** Permitió segmentar objetivamente el 100 % del padrón (**2.744 donantes**), identificando **940 recurrentes (34,26 %)** que sostienen la operación y **1.126 esporádicos/único aporte (41,03 %)** listos para fidelizar sin inventar causas personales.
2. **Modelo Predictivo de Llamadas de Alta Probabilidad:** `RandomForestClassifier` (`ROC-AUC > 0,80`, entrenado sin fuga de datos y validado frente a líneas base) prioriza a quién debe llamar cada **empleada telemercaderista**, evitando saturar a hogares con teléfonos compartidos (**156 grupos reales + caso de los 4 hermanos**).
3. **Consolidado Integral de Información:** Las 130 columnas ruidosas del Excel quedaron integradas sin alterar la fuente (`SHA-256`) en **Parquet + DuckDB** (**20.804 aportes = COP \$3.020 millones**), alimentando el tablero de **11 KPIs**, el **batch diario del mensajero** y los controles de seguridad **RBAC / Ley 1581 de 2012**.